Cell 1 — Environment Setup & GPU Check
Goal: Install all required libraries and verify the GPU is active before doing anything else.
What it does: Installs Ultralytics and timm, clones the Ultralytics repo so we can inject our custom modules into it, forces Python to use the cloned version, then confirms CUDA is available.

In [ ]:
import os, sys
os.chdir('/content')

# Install dependencies
!pip install timm -q

# Clone Ultralytics for custom module injection
if not os.path.exists('/content/ultralytics'):
    !git clone https://github.com/ultralytics/ultralytics.git /content/ultralytics

# Force Python to use the cloned version
sys.path.insert(0, '/content/ultralytics')
!pip install -e "/content/ultralytics[dev]" -q

# Verify GPU
import torch
assert torch.cuda.is_available(), "❌ GPU not found — go to Runtime > Change runtime type > T4 GPU"
print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
print(f"✅ Ultralytics path: {__import__('ultralytics').__file__}")

Cloning into '/content/ultralytics'...
remote: Enumerating objects: 110510, done.
remote: Counting objects: 100% (981/981), done.
remote: Compressing objects: 100% (474/474), done.
remote: Total 110510 (delta 668), reused 575 (delta 506), pack-reused 109529 (from 2)
Receiving objects: 100% (110510/110510), 58.25 MiB | 24.47 MiB/s, done.
Resolving deltas: 100% (83014/83014), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.3/255.3 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

Summary: This cell sets up the entire training environment. It must be the first cell run in every new Colab session. If the assert fails, change the runtime to T4 GPU and restart.


Cell 2 — Mount Google Drive
Goal: Connect the notebook to Google Drive where all data, weights, and results are stored permanently.
What it does: Mounts Drive at /content/drive and verifies the project folder exists.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify project folder exists
BASE = '/content/drive/MyDrive/IR_MODEL'
assert os.path.exists(BASE), f"❌ Project folder not found at {BASE}"
print(f"✅ Drive mounted — project folder found at {BASE}")

Mounted at /content/drive
✅ Drive mounted — project folder found at /content/drive/MyDrive/IR_MODEL


Summary: All files (dataset, weights, results) live on Drive so nothing is lost when Colab disconnects. This cell must run before any file access.

Cell 3 — Inject Custom Modules (CFAN, SWN, TCN)
Goal: Copy the custom RT-DETR-CST module file into Ultralytics and register the three novel modules so the trainer can find them.
What it does: Copies rt_detr_cst_modules.py from Drive into the cloned Ultralytics directory, then patches __init__.py and tasks.py to register CFAN, SWN, and TCN.

In [ ]:
import shutil

# ── Step 1: Copy module file ──────────────────────────
src = '/content/drive/MyDrive/IR_MODEL/modules/rt_detr_cst_modules.py'
dst = '/content/ultralytics/ultralytics/nn/modules/rt_detr_cst.py'
shutil.copy(src, dst)
print(f"✅ Module file copied to {dst}")

# ── Step 2: Patch __init__.py ─────────────────────────
init_path = '/content/ultralytics/ultralytics/nn/modules/__init__.py'
with open(init_path, 'r') as f:
    content = f.read()
line = "from .rt_detr_cst import CFAN, SWN, TCN\n"
if line not in content:
    with open(init_path, 'w') as f:
        f.write(line + content)
    print("✅ __init__.py patched")
else:
    print("⚠️  __init__.py already patched")

# ── Step 3: Patch tasks.py ────────────────────────────
tasks_path = '/content/ultralytics/ultralytics/nn/tasks.py'
with open(tasks_path, 'r') as f:
    content = f.read()
line2 = "from ultralytics.nn.modules.rt_detr_cst import CFAN, SWN, TCN\n"
if line2 not in content:
    with open(tasks_path, 'w') as f:
        f.write(line2 + content)
    print("✅ tasks.py patched")
else:
    print("⚠️  tasks.py already patched")

# ── Step 4: Verify imports ────────────────────────────
from ultralytics.nn.modules.rt_detr_cst import CFAN, SWN, TCN
print("✅ CFAN, SWN, TCN imported successfully")

✅ Module file copied to /content/ultralytics/ultralytics/nn/modules/rt_detr_cst.py
✅ __init__.py patched
✅ tasks.py patched
✅ CFAN, SWN, TCN imported successfully


Summary: This cell injects the three novel modules from the paper into the Ultralytics framework. It must run every new session because Colab resets the file system. The patches are skipped automatically if already applied.



Cell 4.1 — Delete empty folder and redo conversion
What it does: Deletes the empty augmented folder so Cell 4 runs the full conversion again instead of skipping it.

In [ ]:
import shutil, os

aug_path = '/content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented'

# Delete empty augmented folder
if os.path.exists(aug_path):
    shutil.rmtree(aug_path)
    print("✅ Empty folder deleted")
else:
    print("⚠️ Folder not found")

✅ Empty folder deleted


Cell 4 — Convert Dataset (VOC → YOLO Format)
Goal: Convert the ISDD dataset from Pascal VOC XML format to YOLO txt format, then apply infrared-specific augmentations to expand the training set from ~765 to ~3,800 images.
What it does: Reads each XML annotation, converts bounding boxes to normalized YOLO format, copies images, then generates 4 augmented versions per image (horizontal flip, CLAHE contrast enhancement, brightness boost, 90° rotation).

In [ ]:
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import os, shutil

# ── Paths ─────────────────────────────────────────────
BASE_DATA  = '/content/drive/MyDrive/IR_MODEL/ISDD_Dataset'
ANN_DIR    = f'{BASE_DATA}/Annotations'
IMG_DIR    = f'{BASE_DATA}/JPEGImages'
SETS_DIR   = f'{BASE_DATA}/ImageSets'     # note: typo in original dataset

OUT_BASE   = '/content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented'
ORIG_BASE  = '/content/drive/MyDrive/IR_MODEL/ISDD_Dataset/ISDD_YOLO_ready/dataset'

# ── Skip if already done ──────────────────────────────
if os.path.exists(f'{OUT_BASE}/images/train'):
    total = len(os.listdir(f'{OUT_BASE}/images/train'))
    print(f"✅ Augmented dataset already exists ({total} train images) — skipping conversion")
else:
    # Create output folders
    for split in ['train', 'val', 'test']:
        os.makedirs(f'{OUT_BASE}/images/{split}', exist_ok=True)
        os.makedirs(f'{OUT_BASE}/labels/{split}', exist_ok=True)

    def voc_to_yolo(xml_path, img_w, img_h):
        tree = ET.parse(xml_path)
        root = tree.getroot()
        lines = []
        for obj in root.findall('object'):
            bbox = obj.find('bndbox')
            xmin = float(bbox.find('xmin').text)
            ymin = float(bbox.find('ymin').text)
            xmax = float(bbox.find('xmax').text)
            ymax = float(bbox.find('ymax').text)
            cx = (xmin + xmax) / 2 / img_w
            cy = (ymin + ymax) / 2 / img_h
            bw = (xmax - xmin) / img_w
            bh = (ymax - ymin) / img_h
            lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        return '\n'.join(lines)

    def augment_and_save(img, labels, img_id, split):
        out_i = f'{OUT_BASE}/images/{split}'
        out_l = f'{OUT_BASE}/labels/{split}'

        # Original
        cv2.imwrite(f'{out_i}/{img_id}.png', img)
        with open(f'{out_l}/{img_id}.txt', 'w') as f:
            f.write(labels)

        if split != 'train' or not labels.strip():
            return

        lines = [l for l in labels.split('\n') if l.strip()]

        # 1. Horizontal flip
        flipped = cv2.flip(img, 1)
        new_lbls = []
        for l in lines:
            p = l.split()
            new_lbls.append(f"0 {1-float(p[1]):.6f} {p[2]} {p[3]} {p[4]}")
        cv2.imwrite(f'{out_i}/{img_id}_fliph.png', flipped)
        with open(f'{out_l}/{img_id}_fliph.txt', 'w') as f:
            f.write('\n'.join(new_lbls))

        # 2. CLAHE contrast enhancement
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape)==3 else img
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        img_clahe = clahe.apply(gray)
        cv2.imwrite(f'{out_i}/{img_id}_clahe.png', img_clahe)
        with open(f'{out_l}/{img_id}_clahe.txt', 'w') as f:
            f.write(labels)

        # 3. Brightness boost
        img_bright = np.clip(img.astype(np.float32) * 1.5, 0, 255).astype(np.uint8)
        cv2.imwrite(f'{out_i}/{img_id}_bright.png', img_bright)
        with open(f'{out_l}/{img_id}_bright.txt', 'w') as f:
            f.write(labels)

        # 4. Rotation 90°
        img_rot = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
        new_lbls = []
        for l in lines:
            p = l.split()
            new_lbls.append(f"0 {1-float(p[2]):.6f} {float(p[1]):.6f} {p[4]} {p[3]}")
        cv2.imwrite(f'{out_i}/{img_id}_rot90.png', img_rot)
        with open(f'{out_l}/{img_id}_rot90.txt', 'w') as f:
            f.write('\n'.join(new_lbls))

    # Process all splits
    for split in ['train', 'val', 'test']:
        txt_file = f'{SETS_DIR}/{split}.txt'
        with open(txt_file, 'r') as f:
            ids = [l.strip() for l in f if l.strip()]

        ok = skip = 0
        for img_id in ids:
            xml_path = f'{ANN_DIR}/{img_id}.xml'
            img_path = None
            for ext in ['.png', '.jpg', '.jpeg']:
                if os.path.exists(f'{IMG_DIR}/{img_id}{ext}'):
                    img_path = f'{IMG_DIR}/{img_id}{ext}'
                    break

            if not img_path or not os.path.exists(xml_path):
                skip += 1
                continue

            tree = ET.parse(xml_path)
            root = tree.getroot()
            img_w = int(root.find('size/width').text)
            img_h = int(root.find('size/height').text)
            labels = voc_to_yolo(xml_path, img_w, img_h)
            img = cv2.imread(img_path)
            augment_and_save(img, labels, img_id, split)
            ok += 1

        print(f"{split}: ✅ {ok} processed | ⚠️ {skip} skipped")

    print(f"\n✅ Total train images: {len(os.listdir(f'{OUT_BASE}/images/train'))}")

train: ✅ 765 processed | ⚠️ 0 skipped
val: ✅ 134 processed | ⚠️ 0 skipped
test: ✅ 385 processed | ⚠️ 0 skipped

✅ Total train images: 3825


Summary: Converts the ISDD dataset from Pascal VOC to YOLO format and applies 4 infrared-specific augmentations to the training split only. The val and test splits remain unaugmented for honest evaluation. Skips automatically if already done.

Cell 5 — Create Dataset YAML
Goal: Create the configuration file that tells Ultralytics where the dataset is and how many classes it has.

In [ ]:
yaml_content = f"""
path: {OUT_BASE}
train: images/train
val:   images/val
test:  images/test
nc: 1
names: ['ship']
"""

yaml_path = f'{OUT_BASE}/dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("✅ dataset.yaml created at:")
print(yaml_path)
print(yaml_content)

✅ dataset.yaml created at:
/content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/dataset.yaml

path: /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented
train: images/train
val:   images/val
test:  images/test
nc: 1
names: ['ship']



Summary: Creates the YAML config file pointing to the augmented dataset. The val and test sets point to the original unaugmented images for fair evaluation.



Cell 6 — Training (with Auto-Resume)
Goal: Train RT-DETR-CST with pretrained COCO weights, infrared-optimized hyperparameters, and automatic resume capability in case Colab disconnects.
What it does: Checks if a previous checkpoint exists on Drive — if yes, resumes from it; if no, starts fresh from pretrained weights.

In [ ]:
from ultralytics import RTDETR
import os, glob

# ── Define paths (replaces Cell 4 variables) ─────────
OUT_BASE    = '/content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented'
YAML        = f'{OUT_BASE}/dataset.yaml'

# ── Auto-detect checkpoint ────────────────────────────
pts = glob.glob('/content/drive/MyDrive/IR_MODEL/runs/**/last.pt', recursive=True)

if pts:
    LAST_PT = pts[0]
    print(f"🔄 Resuming from: {LAST_PT}")
    model = RTDETR(LAST_PT)
    model.train(
        data     = YAML,
        resume   = True,
        exist_ok = True,
        device   = 0,
    )
else:
    print("🚀 No checkpoint found — starting fresh")
    model = RTDETR('rtdetr-l.pt')
    model.train(
        data           = YAML,
        imgsz          = 640,
        batch          = 4,
        epochs         = 1000,
        patience       = 100,
        optimizer      = 'AdamW',
        lr0            = 0.0001,
        lrf            = 0.01,
        cos_lr         = True,
        warmup_epochs  = 20,
        warmup_bias_lr = 0.001,
        weight_decay   = 0.0005,
        hsv_h          = 0.0,
        hsv_s          = 0.0,
        hsv_v          = 0.4,
        fliplr         = 0.5,
        flipud         = 0.3,
        mosaic         = 0.0,
        mixup          = 0.0,
        scale          = 0.3,
        translate      = 0.1,
        degrees        = 15.0,
        pretrained     = True,
        freeze         = 10,
        device         = 0,
        project        = '/content/drive/MyDrive/IR_MODEL/runs',
        name           = 'rt_detr_cst',
        exist_ok       = True,
        verbose        = True,
    )

🔄 Resuming from: /content/drive/MyDrive/IR_MODEL/runs/rt_detr_cst/weights/last.pt
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/dataset.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1000, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/IR_MODEL/runs/rt_detr_cst/

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    36/1000      2.31G     0.4376     0.4027    0.06045          2        640: 100% ━━━━━━━━━━━━ 957/957 2.4s/it 38:34
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 1.6s/it 26.9s
                   all        134        303      0.909      0.871      0.894      0.427

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    37/1000       2.7G     0.4835     0.4181    0.06276          8        640: 0% ──────────── 0/957  0.4s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    37/1000       2.7G     0.4307     0.4007    0.05952          3        640: 100% ━━━━━━━━━━━━ 957/957 2.8it/s 5:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 5.8it/s 2.9s
                   all        134        303      0.919      0.884      0.901      0.438

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    38/1000       2.7G     0.3313     0.3839    0.06086          4        640: 0% ──────────── 0/957  0.3s

...

    57/1000       2.7G     0.4233     0.3977    0.05764          2        640: 100% ━━━━━━━━━━━━ 957/957 2.9it/s 5:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 5.7it/s 3.0s
                   all        134        303      0.928      0.891      0.897      0.413

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    58/1000       2.7G     0.4091     0.3771    0.06091         25        640: 0% ──────────── 0/957  0.4s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    58/1000       2.7G     0.4224     0.3987    0.05734          3        640: 100% ━━━━━━━━━━━━ 957/957 2.9it/s 5:29
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 5.8it/s 2.9s
                   all        134        303      0.913      0.871      0.879      0.418

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    59/1000       2.7G      0.425     0.3937    0.05631         13        640: 0% ──────────── 0/957  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    59/1000       2.7G     0.4243     0.3982    0.05757         11        640: 79% ━━━━━━━━━╸── 759/957 2.4it/s 4:22<1:22

Summary: This is the main training cell. It automatically detects whether a previous checkpoint exists and resumes from it — no manual intervention needed after a Colab disconnect. Just re-run cells 1 through 6 in order.



Cell 7 — Evaluate on Test Set
Goal: Load the best saved weights and measure final performance on the unseen test set.

In [ ]:
from ultralytics import RTDETR

BEST_PT = '/content/drive/MyDrive/IR_MODEL/runs/rt_detr_cst/weights/best.pt'
assert os.path.exists(BEST_PT), "❌ best.pt not found — training may not have completed yet"

model = RTDETR(BEST_PT)
metrics = model.val(
    data   = f'{OUT_BASE}/dataset.yaml',
    split  = 'test',
    device = 0,
)

print("─" * 40)
print(f"mAP@0.5     : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision   : {metrics.box.mp:.4f}")
print(f"Recall      : {metrics.box.mr:.4f}")
print("─" * 40)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 313 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 0.4±0.2 MB/s, size: 165.7 KB)
val: Scanning /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/labels/test... 385 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 385/385 3.6it/s 1:47
val: New cache created: /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 1.4it/s 17.8s
                   all        385        959      0.923      0.887      0.933      0.433
Speed: 1.3ms preprocess, 38.9ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/ultralytics/runs/detect/val
────────────────────────────────────────
mAP@0.5     : 0.9327
mAP@0.5:0.95: 0.4329
Precision   : 0.9234
Recall      : 0.8874
────────────────────

Summary: Evaluates the best checkpoint on the held-out test set. Run this only after training completes or is far enough along to be meaningful.



Cell 8 — Run Inference on Test Images
Goal: Visually verify the model's detection quality by running inference on sample test images and saving the results to Drive.

In [ ]:
from ultralytics import RTDETR
import glob

BEST_PT   = '/content/drive/MyDrive/IR_MODEL/runs/rt_detr_cst/weights/best.pt'
TEST_IMGS = f'{OUT_BASE}/images/test'
OUT_DIR   = '/content/drive/MyDrive/IR_MODEL/predictions'
os.makedirs(OUT_DIR, exist_ok=True)

model = RTDETR(BEST_PT)

# Run on first 20 test images
test_images = sorted(glob.glob(f'{TEST_IMGS}/*.png'))[:20]

results = model.predict(
    source  = test_images,
    conf    = 0.3,
    device  = 0,
    save    = True,
    project = OUT_DIR,
    name    = 'test_predictions',
    exist_ok= True,
)

print(f"✅ Predictions saved to {OUT_DIR}/test_predictions/")
print(f"   Total images processed: {len(results)}")


image 1/20 /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/images/test/000004.png: 640x640 2 ships, 71.1ms
image 2/20 /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/images/test/000010.png: 640x640 3 ships, 59.6ms
image 3/20 /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/images/test/000011.png: 640x640 1 ship, 60.5ms
image 4/20 /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/images/test/000017.png: 640x640 2 ships, 108.6ms
image 5/20 /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/images/test/000018.png: 640x640 2 ships, 60.1ms
image 6/20 /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/images/test/000020.png: 640x640 2 ships, 57.7ms
image 7/20 /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/images/test/000026.png: 640x640 2 ships, 57.7ms
image 8/20 /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/images/test/000028.png: 640x640 1 ship, 51.3ms
image 9/20 /content/drive/MyDrive/IR_MODEL/ISDD_Dataset/augmented/images/test/000033.png

Summary: Runs the trained model on 20 test images and saves annotated outputs to Drive. Open the saved images in Drive to visually inspect detection quality — bounding boxes and confidence scores will be drawn on each image.



In [ ]:
from ultralytics import RTDETR
import os

# Define paths (reusing variables from previous cells)
BEST_PT = '/content/drive/MyDrive/IR_MODEL/runs/rt_detr_cst/weights/best.pt'
OUT_DIR = '/content/drive/MyDrive/IR_MODEL/predictions'

# Ensure output directory exists
os.makedirs(OUT_DIR, exist_ok=True)

# --- USER ACTION REQUIRED: UPLOAD YOUR IMAGE TO DRIVE AND UPDATE THIS PATH ---
# Example: If you upload `my_ship_image.png` to your 'IR_MODEL' folder
INPUT_IMAGE_PATH = '/content/drive/MyDrive/IR_MODEL/my_ship_image.png'
# -----------------------------------------------------------------------------

# Check if the input image path exists (optional, but good practice)
if not os.path.exists(INPUT_IMAGE_PATH):
    print(f"❌ Input image not found at: {INPUT_IMAGE_PATH}")
    print("Please upload your image to Google Drive and update the `INPUT_IMAGE_PATH` variable.")
else:
    print(f"✅ Input image found at: {INPUT_IMAGE_PATH}")
    model = RTDETR(BEST_PT)

    # Run inference on the single image
    results = model.predict(
        source  = INPUT_IMAGE_PATH,
        conf    = 0.3, # Confidence threshold
        device  = 0,   # Use GPU if available
        save    = True, # Save results with bounding boxes
        project = OUT_DIR,
        name    = 'single_image_prediction', # Subfolder name for this prediction run
        exist_ok= True,
    )

    print(f"\n✅ Prediction saved to {OUT_DIR}/single_image_prediction/")
    print(f"   Processed image: {INPUT_IMAGE_PATH}")


In [ ]:
jvjvcjvj